# Biohub - Cell Tracking During Development
## Score: 0.880

## Configuration

In [ ]:
import os
from pathlib import Path

COMP_DIR = "/kaggle/input/competitions/biohub-cell-tracking-during-development"
TEST_DIR = f"{COMP_DIR}/test"
ARTIFACTS_ROOT = Path(
    "/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts"
)
WEIGHTS_CANDIDATES = [
    Path("/kaggle/input/datasets/hongdaekim/biohub-350ep-checkpoint-pin-v1"),
    Path("/kaggle/input/hongdaekim/biohub-350ep-checkpoint-pin-v1"),
    Path("/kaggle/input/biohub-350ep-checkpoint-pin-v1"),
    Path(
        "/kaggle/input/datasets/hongdaekim/"
        "biohub-350ep-checkpoint-pin-v1/biohub-350ep-checkpoint-pin-v1"
    ),
]
REPO_DIR = "/kaggle/working/repo"
METHOD = "unet_transformer"
WEIGHTS = f"weights/{METHOD}/split_0/edge_predictor_best.pth"
DET_THRESHOLD = 0.99
UNET_BATCH_SIZE = 4
USE_ILP = True
ILP_EDGE_WEIGHT = -1.0
ILP_APPEARANCE_WEIGHT = 0.1
ILP_DISAPPEARANCE_WEIGHT = 0.1
ILP_DIVISION_WEIGHT = 1.0
MOTION_RELINK_TIGHT_UM = 6.0
MOTION_RELINK_RELAXED_UM = 10.0
MOTION_RELINK_VELOCITY_WEIGHT = 0.5
MOTION_RELINK_LEARNED_BONUS = 1.0
MOTION_RELINK_MAX_FRAME_NODES = 2600
MIN_TRACK_LEN = 6
KEEP_DIVISION_COMPONENTS = True
ADAPTIVE_SHORT_TRACK_RESCUE = True
SHORT_TRACK_RESCUE_MIN_LEN = 3
SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC = 0.05
SHORT_TRACK_RESCUE_MAX_NODES_FRAC = 0.02
SHORT_TRACK_RESCUE_MAX_NODES_ABS = 400
SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB = 0.35
SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM = 6.0
OUTPUT_PATH = "/kaggle/working/submission.csv"

if (ARTIFACTS_ROOT / "repo").exists():
    ARTIFACTS = ARTIFACTS_ROOT
elif (ARTIFACTS_ROOT / "cellmot-baseline-artifacts" / "repo").exists():
    ARTIFACTS = ARTIFACTS_ROOT / "cellmot-baseline-artifacts"
else:
    ARTIFACTS = ARTIFACTS_ROOT

WEIGHTS_SRC = next(
    (
        path
        for path in WEIGHTS_CANDIDATES
        if (path / "edge_predictor_best.pth").exists()
    ),
    None,
)
if WEIGHTS_SRC is None:
    for path in Path("/kaggle/input").rglob("edge_predictor_best.pth"):
        if "thibautgoldsborough" in str(path):
            continue
        WEIGHTS_SRC = path.parent
        break

COMP_DIR, ARTIFACTS, WEIGHTS_SRC, OUTPUT_PATH


## Offline Install

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

assert WEIGHTS_SRC is not None, (
    "350ep checkpoint not found under /kaggle/input. "
    "Attach hongdaekim/biohub-350ep-checkpoint-pin-v1 and re-run."
)
assert (ARTIFACTS / "wheels").exists(), f"Missing wheels at {ARTIFACTS / 'wheels'}"
assert (ARTIFACTS / "repo").exists(), f"Missing repo at {ARTIFACTS / 'repo'}"

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-index",
        "--find-links",
        str(ARTIFACTS / "wheels"),
        "--upgrade-strategy",
        "only-if-needed",
        "tracksdata",
        "zarr>=3.0.10",
        "pyscipopt",
    ],
    check=True,
)

shutil.copytree(ARTIFACTS / "repo", REPO_DIR, dirs_exist_ok=True)
if (ARTIFACTS / "weights").exists():
    shutil.copytree(ARTIFACTS / "weights", Path(REPO_DIR) / "weights", dirs_exist_ok=True)

weights_dir = Path(REPO_DIR) / "weights" / METHOD / "split_0"
weights_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(WEIGHTS_SRC / "edge_predictor_best.pth", weights_dir / "edge_predictor_best.pth")
config_src = WEIGHTS_SRC / "config.json"
if config_src.exists():
    shutil.copy2(config_src, weights_dir / "config.json")

sys.path.insert(0, f"{REPO_DIR}/src")
WEIGHTS_SRC, sorted(weights_dir.iterdir()), (weights_dir / "edge_predictor_best.pth").stat().st_size


## Test Split

In [ ]:
import json
from pathlib import Path

test_stems = sorted(
    path.name.replace(".zarr", "")
    for path in Path(TEST_DIR).glob("*.zarr")
)
splits_path = Path(REPO_DIR) / "kaggle_test_splits.json"
splits_path.write_text(
    json.dumps([{"split": 0, "train": [], "test": test_stems}])
)
len(test_stems), test_stems[:5]

## Inference

In [ ]:
import os
import subprocess

cmd = [
    sys.executable,
    "scripts/predict_unet_transformer.py",
    "--data-dir",
    TEST_DIR,
    "--splits",
    "kaggle_test_splits.json",
    "--split",
    "0",
    "--weights",
    WEIGHTS,
    "--unet-batch-size",
    str(UNET_BATCH_SIZE),
    "--det-threshold",
    str(DET_THRESHOLD),
    "--ilp-edge-weight",
    str(ILP_EDGE_WEIGHT),
    "--ilp-appearance-weight",
    str(ILP_APPEARANCE_WEIGHT),
    "--ilp-disappearance-weight",
    str(ILP_DISAPPEARANCE_WEIGHT),
    "--ilp-division-weight",
    str(ILP_DIVISION_WEIGHT),
]
if USE_ILP:
    cmd.append("--use-ilp")

print(" ".join(cmd))
subprocess.run(
    cmd,
    cwd=REPO_DIR,
    env={**os.environ, "PYTHONPATH": "src"},
    check=True,
)

## Graph Repair


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

pred_dirs = sorted(Path(REPO_DIR, "predictions").glob(f"*/{METHOD}/split_0"))
assert pred_dirs, "No prediction directory found"
PRED_DIR = pred_dirs[0]

repair_code = r"""
import csv
import math
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import zarr
from scipy.optimize import linear_sum_assignment

pred_dir = Path(sys.argv[1])
out_csv = Path(sys.argv[2])
tight_um = float(sys.argv[3])
relaxed_um = float(sys.argv[4])
velocity_weight = float(sys.argv[5])
learned_bonus = float(sys.argv[6])
max_frame_nodes = int(sys.argv[7])
min_track_len = int(sys.argv[8])
keep_division_components = sys.argv[9] == "1"
adaptive_rescue = sys.argv[10] == "1"
rescue_min_len = int(sys.argv[11])
rescue_trigger_frac = float(sys.argv[12])
rescue_max_frac = float(sys.argv[13])
rescue_max_abs = int(sys.argv[14])
rescue_min_prob = float(sys.argv[15])
rescue_max_dist = float(sys.argv[16])
scale = np.array([1.625, 0.40625, 0.40625], dtype=np.float64)


def load_graph(path: Path):
    root = zarr.open(str(path), mode="r")
    node_ids = np.asarray(root["nodes"]["ids"][:])
    t = np.asarray(root["nodes"]["props"]["t"]["values"][:])
    z = np.asarray(root["nodes"]["props"]["z"]["values"][:])
    y = np.asarray(root["nodes"]["props"]["y"]["values"][:])
    x = np.asarray(root["nodes"]["props"]["x"]["values"][:])
    if "solution" in root["nodes"]["props"]:
        keep = np.asarray(root["nodes"]["props"]["solution"]["values"][:]).astype(bool)
    else:
        keep = np.ones(len(node_ids), dtype=bool)
    nodes = {}
    for i, node_id in enumerate(node_ids):
        if not keep[i]:
            continue
        nodes[int(node_id)] = {
            "node_id": int(node_id),
            "t": int(t[i]),
            "z": float(z[i]),
            "y": float(y[i]),
            "x": float(x[i]),
        }
    edges_raw = np.asarray(root["edges"]["ids"][:])
    if "solution" in root["edges"]["props"]:
        edge_keep = np.asarray(root["edges"]["props"]["solution"]["values"][:]).astype(bool)
    else:
        edge_keep = np.ones(len(edges_raw), dtype=bool)
    props = root["edges"]["props"]
    prob_arr = None
    for key in ("edge_prob", "score", "prob", "weight"):
        if key in props:
            prob_arr = np.asarray(props[key]["values"][:])
            break
    edges = []
    edge_probs = {}
    for i, (source, target) in enumerate(edges_raw):
        if not edge_keep[i]:
            continue
        source, target = int(source), int(target)
        if source not in nodes or target not in nodes:
            continue
        prob = float(prob_arr[i]) if prob_arr is not None else 0.0
        edges.append({"source_id": source, "target_id": target, "edge_prob": prob})
        edge_probs[(source, target)] = max(edge_probs.get((source, target), float("-inf")), prob)
    return nodes, edges, edge_probs


def position_um(node):
    return np.array(
        [node["z"] * scale[0], node["y"] * scale[1], node["x"] * scale[2]],
        dtype=np.float64,
    )


def learned_prob(edge_probs, source_id, target_id):
    value = edge_probs.get((source_id, target_id), 0.0)
    try:
        value = float(value)
    except (TypeError, ValueError):
        return 0.0
    if not np.isfinite(value):
        return 0.0
    if value < 0.0 or value > 1.0:
        value = 1.0 / (1.0 + math.exp(-max(-20.0, min(20.0, value))))
    return float(np.clip(value, 0.0, 1.0))


def motion_rescue(nodes, edges, edge_probs, stats):
    if not nodes:
        return edges
    ids_by_t = defaultdict(list)
    for node_id, node in nodes.items():
        ids_by_t[int(node["t"])].append(node_id)
    for ids in ids_by_t.values():
        ids.sort()
    if ids_by_t and max(len(v) for v in ids_by_t.values()) > max_frame_nodes:
        stats["motion_relink_skipped_large_frame"] = 1
        return edges

    pos = {node_id: position_um(node) for node_id, node in nodes.items()}
    enriched = []
    for edge in edges:
        s = nodes[edge["source_id"]]
        t = nodes[edge["target_id"]]
        raw = float(np.linalg.norm(pos[edge["source_id"]] - pos[edge["target_id"]]))
        item = dict(edge)
        item["distance_um"] = raw
        enriched.append(item)
    edges = enriched

    parent_of = {}
    children_of = defaultdict(list)
    pred_pos = {}
    for edge in edges:
        s, t = int(edge["source_id"]), int(edge["target_id"])
        parent_of[t] = s
        children_of[s].append(t)
        pred_pos[t] = pos[s]

    def assign_pass(source_ids, target_ids, gate_um):
        if not source_ids or not target_ids:
            return []
        big = gate_um * 1000.0 + 1.0
        cost = np.full((len(source_ids), len(target_ids)), big, dtype=np.float64)
        raw_dist = np.full_like(cost, np.inf)
        motion_dist = np.full_like(cost, np.inf)
        prob_matrix = np.zeros_like(cost)
        for i, source_id in enumerate(source_ids):
            source_pos = pos[source_id]
            prev = pred_pos.get(source_id)
            predicted = (
                source_pos
                if prev is None
                else source_pos + velocity_weight * (source_pos - prev)
            )
            for j, target_id in enumerate(target_ids):
                target_pos = pos[target_id]
                raw = float(np.linalg.norm(target_pos - source_pos))
                if raw > gate_um:
                    continue
                motion = float(np.linalg.norm(target_pos - predicted))
                prob = learned_prob(edge_probs, source_id, target_id)
                raw_dist[i, j] = raw
                motion_dist[i, j] = motion
                prob_matrix[i, j] = prob
                cost[i, j] = motion + 0.05 * raw - learned_bonus * prob
        rows, cols = linear_sum_assignment(cost)
        matches = []
        for r, c in zip(rows, cols):
            if cost[r, c] >= big:
                continue
            matches.append(
                (
                    source_ids[int(r)],
                    target_ids[int(c)],
                    float(raw_dist[r, c]),
                    float(prob_matrix[r, c]),
                )
            )
        return matches

    for t in sorted(ids_by_t):
        source_ids = [
            i
            for i in ids_by_t.get(t, [])
            if len(children_of[i]) == 0
        ]
        target_ids = [
            i
            for i in ids_by_t.get(t + 1, [])
            if i not in parent_of
        ]
        if not source_ids or not target_ids:
            continue
        unmatched_s = set(source_ids)
        unmatched_t = set(target_ids)
        for pass_name, gate in (("tight", tight_um), ("relaxed", relaxed_um)):
            s_ids = [i for i in source_ids if i in unmatched_s]
            t_ids = [i for i in target_ids if i in unmatched_t]
            for source_id, target_id, raw, prob in assign_pass(s_ids, t_ids, gate):
                if source_id not in unmatched_s or target_id not in unmatched_t:
                    continue
                if len(children_of[source_id]) >= 1:
                    continue
                if target_id in parent_of:
                    continue
                unmatched_s.remove(source_id)
                unmatched_t.remove(target_id)
                edges.append(
                    {
                        "source_id": source_id,
                        "target_id": target_id,
                        "edge_prob": prob,
                        "distance_um": raw,
                    }
                )
                children_of[source_id].append(target_id)
                parent_of[target_id] = source_id
                pred_pos[target_id] = pos[source_id]
                stats[f"motion_relink_{pass_name}_edges"] += 1
                stats["motion_relink_edges"] += 1
        stats["motion_relink_frames"] += 1
    return edges


def filter_short_tracks(nodes, edges, stats):
    if min_track_len <= 1 or not edges:
        return nodes, edges
    parent = {node_id: node_id for node_id in nodes}

    def find(node_id):
        while parent[node_id] != node_id:
            parent[node_id] = parent[parent[node_id]]
            node_id = parent[node_id]
        return node_id

    def union(a, b):
        if a not in parent or b not in parent:
            return
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    out_count = defaultdict(int)
    for edge in edges:
        s, t = int(edge["source_id"]), int(edge["target_id"])
        union(s, t)
        out_count[s] += 1
    components = defaultdict(list)
    for node_id in nodes:
        components[find(node_id)].append(node_id)
    component_edges = defaultdict(list)
    for edge in edges:
        s, t = int(edge["source_id"]), int(edge["target_id"])
        if s in parent and t in parent:
            component_edges[find(s)].append(edge)
    keep = set()
    for root, members in components.items():
        has_div = any(out_count[n] >= 2 for n in members)
        if len(members) >= min_track_len or (keep_division_components and has_div):
            keep.update(members)
    if not keep:
        stats["short_track_filter_skipped_all"] = 1
        return nodes, edges
    removed_before = len(nodes) - len(keep)
    if removed_before <= 0:
        return nodes, edges
    if adaptive_rescue:
        removed_frac = removed_before / max(len(nodes), 1)
        if removed_frac >= rescue_trigger_frac:
            budget = min(rescue_max_abs, max(0, int(round(len(nodes) * rescue_max_frac))))
            stats["short_track_rescue_triggered"] = 1
            stats["short_track_rescue_budget"] = budget
            proposals = []
            for root, members in components.items():
                if set(members) & keep:
                    continue
                if len(members) < rescue_min_len or len(members) >= min_track_len:
                    continue
                c_edges = component_edges.get(root, [])
                if not c_edges:
                    continue
                probs = []
                dists = []
                for edge in c_edges:
                    try:
                        probs.append(float(edge.get("edge_prob", 0.0)))
                    except (TypeError, ValueError):
                        pass
                    try:
                        dists.append(float(edge.get("distance_um", np.nan)))
                    except (TypeError, ValueError):
                        pass
                mean_prob = float(np.mean(probs)) if probs else 0.0
                finite_dists = [d for d in dists if np.isfinite(d)]
                mean_dist = float(np.mean(finite_dists)) if finite_dists else float("inf")
                if mean_prob < rescue_min_prob or mean_dist > rescue_max_dist:
                    continue
                score = mean_prob - 0.02 * mean_dist + 0.004 * len(members)
                proposals.append((score, len(members), members))
            proposals.sort(reverse=True)
            rescued_nodes = 0
            rescued_components = 0
            for _, size, members in proposals:
                if budget <= 0 or rescued_nodes + size > budget:
                    continue
                keep.update(members)
                rescued_nodes += size
                rescued_components += 1
            stats["short_track_rescue_components"] = rescued_components
            stats["short_track_rescue_nodes"] = rescued_nodes
    kept_nodes = {i: n for i, n in nodes.items() if i in keep}
    kept_edges = [
        e
        for e in edges
        if int(e["source_id"]) in kept_nodes and int(e["target_id"]) in kept_nodes
    ]
    stats["short_track_nodes_removed"] = len(nodes) - len(kept_nodes)
    stats["short_track_edges_removed"] = len(edges) - len(kept_edges)
    return kept_nodes, kept_edges


rows = []
for geff in sorted(pred_dir.glob("*.geff")):
    nodes, raw_edges, edge_probs = load_graph(geff)
    stats = defaultdict(int)
    edges = motion_rescue(nodes, raw_edges, edge_probs, stats)
    incident = {int(e["source_id"]) for e in edges} | {int(e["target_id"]) for e in edges}
    if incident:
        nodes = {i: n for i, n in nodes.items() if i in incident}
        edges = [
            e
            for e in edges
            if int(e["source_id"]) in nodes and int(e["target_id"]) in nodes
        ]
    nodes, edges = filter_short_tracks(nodes, edges, stats)
    dataset = geff.stem
    for node_id, node in sorted(nodes.items()):
        rows.append(
            [
                dataset,
                "node",
                node_id,
                int(node["t"]),
                int(round(node["z"])),
                int(round(node["y"])),
                int(round(node["x"])),
                -1,
                -1,
            ]
        )
    for edge in edges:
        rows.append(
            [
                dataset,
                "edge",
                -1,
                -1,
                -1,
                -1,
                -1,
                int(edge["source_id"]),
                int(edge["target_id"]),
            ]
        )
    print(dataset, dict(stats), "nodes", len(nodes), "edges", len(edges))

with out_csv.open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(
        [
            "id",
            "dataset",
            "row_type",
            "node_id",
            "t",
            "z",
            "y",
            "x",
            "source_id",
            "target_id",
        ]
    )
    for index, row in enumerate(rows):
        writer.writerow([index, *row])
print("rows", len(rows), "out", out_csv)
"""

cmd = [
    sys.executable,
    "-c",
    repair_code,
    str(PRED_DIR),
    OUTPUT_PATH,
    str(MOTION_RELINK_TIGHT_UM),
    str(MOTION_RELINK_RELAXED_UM),
    str(MOTION_RELINK_VELOCITY_WEIGHT),
    str(MOTION_RELINK_LEARNED_BONUS),
    str(MOTION_RELINK_MAX_FRAME_NODES),
    str(MIN_TRACK_LEN),
    "1" if KEEP_DIVISION_COMPONENTS else "0",
    "1" if ADAPTIVE_SHORT_TRACK_RESCUE else "0",
    str(SHORT_TRACK_RESCUE_MIN_LEN),
    str(SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC),
    str(SHORT_TRACK_RESCUE_MAX_NODES_FRAC),
    str(SHORT_TRACK_RESCUE_MAX_NODES_ABS),
    str(SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB),
    str(SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM),
]
print("repair", PRED_DIR, "->", OUTPUT_PATH)
subprocess.run(cmd, check=True)
Path(OUTPUT_PATH).exists(), Path(OUTPUT_PATH).stat().st_size


## Submission

In [ ]:
from pathlib import Path

assert Path(OUTPUT_PATH).exists(), f"Missing {OUTPUT_PATH}"
Path(OUTPUT_PATH).exists(), Path(OUTPUT_PATH).stat().st_size


## Submission Checks

In [ ]:
import csv
from pathlib import Path

expected = set(test_stems)
datasets = set()
row_count = 0
with Path(OUTPUT_PATH).open(encoding="utf-8") as file:
    reader = csv.DictReader(file)
    assert reader.fieldnames[0] == "id"
    for expected_id, row in enumerate(reader):
        assert int(row["id"]) == expected_id
        datasets.add(row["dataset"])
        row_count += 1

assert datasets == expected
assert row_count > 0
row_count, Path(OUTPUT_PATH).stat().st_size